In [0]:
# Imports

from pyspark.sql.functions import col, trim, to_date, expr, when, current_timestamp
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import col

In [0]:
# Ler a Bronze V2
df_silver = spark.table("workspace.drs_bronze.covid_19")
print(f"Total registros: {df_silver.count()}")
display(df_silver)

In [0]:
# Transformar/Convert colunas para o padarão para depois salvar a Silver V2

from pyspark.sql.functions import col, trim, to_date, expr, when

# Recriando o df_covid_19 com os tipos corretos
df_covid_19 = (
    df_silver
    .withColumn("city_ibge_code", col("city_ibge_code").cast("int"))
    .withColumn("date", to_date(col("date"))) # Removido o formato fixo que dava erro
    .withColumn("estimated_population", col("estimated_population").cast("int"))
    .withColumn("estimated_population_2019", col("estimated_population_2019").cast("int"))
    .withColumn("is_last", col("is_last").cast("boolean"))
    .withColumn("is_repeated", col("is_repeated").cast("boolean"))
)

# Verificação imediata do esquema
df_covid_19.printSchema()

In [0]:
# criando o campo source_file
from pyspark.sql.functions import lit

df = df_covid_19.withColumn("source_file", lit("covid_19_caso_full.csv.gz"))

In [0]:
# Separando registros inválidos para quarentena

df_invalid = df.filter("""
    last_available_confirmed IS NULL
    OR last_available_confirmed_per_100k_inhabitants IS NULL
    OR last_available_date IS NULL
    OR last_available_death_rate IS NULL
""")

print(f"Total registros inválidos: {df_invalid.count()}")
display(df_invalid)

In [0]:
# Salvando quarentena da Silver V2

df_invalid.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.drs_silver.covid_19_quarantine")


In [0]:
# criando o campo ingestion_timestamp
from pyspark.sql.functions import current_timestamp

df = df.withColumn("ingestion_timestamp", current_timestamp())

In [0]:
# Filtrando valores válidos
from pyspark.sql.functions import col

df_silver_filtered = (
    df.dropDuplicates()
      .filter(
          (col("city_ibge_code").isNotNull()) &
          (col("date").isNotNull())
      )
)

print(f"Total registros inválidos: {df_silver_filtered.count()}")
display(df_silver_filtered)


In [0]:
# Criando o campo silver_processed_timestamp em df_silver_v2
from pyspark.sql.functions import current_timestamp

df_silver_filtered = df_silver_filtered \
.withColumn("silver_processed_timestamp", current_timestamp()) \
.withColumn("created_at", current_timestamp()) \
.withColumn("updated_at", current_timestamp())

In [0]:
display(df_silver_filtered)

In [0]:
# Grantindo a deduplicação por chave de negócio

window_spec = Window.partitionBy(
    "city_ibge_code",
    "date",
    "last_available_date",
).orderBy(
    col("ingestion_timestamp").desc()
)

df_silver_deduplicated = (
    df_silver_filtered
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

In [0]:
# Validando shema da dataframe v2
df_silver_deduplicated.printSchema()

In [0]:
# Salvando a tabela Silver em uma staging table

df_silver_deduplicated.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("workspace.drs_silver.covid_19_staging")

In [0]:

# Craindo tabela final se não existir

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.drs_silver.covid_19
AS
SELECT * FROM workspace.drs_silver.covid_19_staging WHERE 1 = 0
""")

In [0]:
# Validar schema da staging

spark.table("workspace.drs_silver.covid_19_staging").printSchema()

In [0]:
# Salvando tabela drs_silver_v2 com MERGE

spark.sql("""
MERGE INTO workspace.drs_silver.covid_19 AS target
USING workspace.drs_silver.covid_19_staging AS source

ON target.city_ibge_code = source.city_ibge_code
AND target.date = source.date

WHEN MATCHED THEN
UPDATE SET
    target.city = source.city,
    target.epidemiological_week = source.epidemiological_week,
    target.estimated_population = source.estimated_population,
    target.estimated_population_2019 = source.estimated_population_2019,
    target.is_last = source.is_last,
    target.is_repeated = source.is_repeated,
    target.last_available_confirmed = source.last_available_confirmed,
    target.last_available_confirmed_per_100k_inhabitants = source.last_available_confirmed_per_100k_inhabitants,
    target.last_available_date = source.last_available_date,
    target.last_available_death_rate = source.last_available_death_rate,
    target.last_available_deaths = source.last_available_deaths,
    target.order_for_place = source.order_for_place,
    target.place_type = source.place_type,
    target.state = source.state,
    target.new_confirmed = source.new_confirmed,
    target.new_deaths = source.new_deaths,
    target.ingestion_timestamp = source.ingestion_timestamp,
    target.source_file = source.source_file,
    target.silver_processed_timestamp = source.silver_processed_timestamp,
    target.updated_at = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    city,
    city_ibge_code,
    date,
    epidemiological_week,
    estimated_population,
    estimated_population_2019,
    is_last,
    is_repeated,
    last_available_confirmed,
    last_available_confirmed_per_100k_inhabitants,
    last_available_date,
    last_available_death_rate,
    last_available_deaths,
    order_for_place,
    place_type,
    state,
    new_confirmed,
    new_deaths,
    ingestion_timestamp,
    source_file,
    silver_processed_timestamp,
    created_at,
    updated_at
)
VALUES (
    source.city,
    source.city_ibge_code,
    source.date,
    source.epidemiological_week,
    source.estimated_population,
    source.estimated_population_2019,
    source.is_last,
    source.is_repeated,
    source.last_available_confirmed,
    source.last_available_confirmed_per_100k_inhabitants,
    source.last_available_date,
    source.last_available_death_rate,
    source.last_available_deaths,
    source.order_for_place,
    source.place_type,
    source.state,
    source.new_confirmed,
    source.new_deaths,
    source.ingestion_timestamp,
    source.source_file,
    source.silver_processed_timestamp,
    current_timestamp(),
    current_timestamp()
)
""")


In [0]:
# Validando a tabela drs_silver_votacao_secao_2024_go

spark.sql("""
SELECT
COUNT(*) AS total_rows
FROM workspace.drs_silver.covid_19
--GROUP BY 1 
""").display()

In [0]:
# Verificar duplicidade pela chave do merge

spark.sql("""
SELECT
    city_ibge_code,
    date,
    last_available_date,
    COUNT(*) AS qtd
FROM workspace.drs_silver.covid_19
GROUP BY 1,2,3 
HAVING COUNT(*) > 1
ORDER BY qtd DESC
""").display()

In [0]:
# Data Quality checks da Silver V2
dq = spark.sql("""
SELECT
    SUM(CASE WHEN new_confirmed < 0  or new_confirmed is null THEN 1 ELSE 0 END) AS new_confirmed,
    SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS date
FROM workspace.drs_silver.covid_19
""")

display(dq)

In [0]:
# validação quarentena
spark.sql("""
SELECT COUNT(*) AS total_invalid
FROM workspace.drs_silver.covid_19_quarantine
""").display()

In [0]:
# Falhar notebook se houver erro crítico de qualidade
dq_row = dq.collect()[0]

if (
    dq_row["vote_count"] > 0 or
    dq_row["ballot_number"] > 0 
):
    raise Exception("Data Quality check failed in workspace.drs_silver.sales_v2")
